# 10 — Known Differences vs PySpark

Demonstrates the documented deviations where IrisPark deliberately differs from PySpark. See `docs/known_differences.md` for the full reference.

In [ ]:
import os
from dotenv import load_dotenv
from irispark import IrisParkSession

load_dotenv()

# Connection via environment variables (matches examples/basic_usage.py).
# Set IRIS_HOST / IRIS_PORT / IRIS_NAMESPACE / IRIS_USERNAME / IRIS_PASSWORD.
try:
    session = IrisParkSession.builder() \
        .host(os.environ.get("IRIS_HOST", "localhost")) \
        .port(int(os.environ.get("IRIS_PORT", 1972))) \
        .namespace(os.environ.get("IRIS_NAMESPACE", "USER")) \
        .username(os.environ.get("IRIS_USERNAME", "_SYSTEM")) \
        .password(os.environ.get("IRIS_PASSWORD", "SYS")) \
        .getOrCreate()
    print("Connected to IRIS:", session)
except Exception as e:
    print("SKIP: IRIS not reachable -", e)
    session = None

In [ ]:
if session is None:
    raise SystemExit("IRIS not reachable; skipping this notebook.")

## 1. `round` at exact `.5` double boundary

IRIS rounds the binary double; Spark rounds the decimal string. `2.675` → `2.67` (IrisPark) vs `2.68` (Spark).

In [ ]:
import pandas as pd
from irispark.functions import round as rnd

df = session.createDataFrame(pd.DataFrame({"x": [2.675]}))
df.select("x", rnd("x", 2).alias("rounded")).show()
print("pandas round(2.675, 2):", round(2.675, 2))

## 2. `pow`/`power` domain errors → NULL

IRIS cannot represent Inf/NaN; `POWER(0,-1)` is guarded to NULL (Spark yields `inf`).

In [ ]:
from irispark.functions import pow
import numpy as np

df = session.createDataFrame(pd.DataFrame({"a": [0.0], "b": [-1.0]}))
df.select("a", "b", pow("a", "b").alias("pow")).show()

# Python's builtin `0.0 ** -1` raises ZeroDivisionError (not representable);
# numpy/Spark yield inf, while IrisPark guards the domain to NULL.
print("numpy 0.0 ** -1 (Spark-like):", np.power(0.0, -1))
try:
    print("python 0.0 ** -1:", 0.0 ** -1)
except ZeroDivisionError as e:
    print("python 0.0 ** -1 raises:", type(e).__name__)

## 3. `regexp_extract` no-match → NULL

IRIS VARCHAR maps the empty string to NULL; Spark returns `''`.

In [ ]:
from irispark.functions import regexp_extract

df = session.createDataFrame(pd.DataFrame({"s": ["abc"]}))
df.select("s", regexp_extract("s", "xyz", 0).alias("extract")).show()

## 4. `StatFunctions.cov` with no valid pair → `None`

Spark returns `0.0`; IrisPark returns `None` (consistent with `corr`).

In [ ]:
df = session.createDataFrame(pd.DataFrame({"x": [None, None], "y": [1.0, 2.0]}))
print("cov:", df.stat.cov("x", "y"))

## 5. Unsupported APIs

Some PySpark APIs are not implemented (arrays/maps/structs, JSON family, `groupingSets`, bitwise ops). Attempting them raises an error.

In [ ]:
try:
    from irispark.functions import array
    print(array("x"))
except Exception as e:
    print("array() not available:", type(e).__name__, e)

## 6. `VectorAssembler` emits a string, not a numeric vector

PySpark's `VectorAssembler` produces a dense `Vector` column (`[31.0,10.0]`). IrisPark emits a comma-joined **string** column (`"31.0,10.0"`) because IRIS has no native vector type. The ML estimators (`LinearRegression`, `LogisticRegression`) consume the raw numeric `featuresCol` columns directly, so results are unaffected — the vector column is illustrative.

In [ ]:
from irispark.ml.feature import VectorAssembler

df = session.createDataFrame([(31, 10)], ["age", "Experience"])
va = VectorAssembler(inputCols=["age", "Experience"], outputCol="features").transform(df)
va.select("features").show()
va.printSchema()

## 7. `writer.jdbc` write-back is best-effort

IRIS's JDBC foreign-table write path has a known internal limitation (`copyFTInformation` error) when the source is a temp table, so `writer.jdbc` may not complete end-to-end. Reads via `read.jdbc`/foreign tables are unaffected.

In [ ]:
try:
    df = session.createDataFrame([(1, "x")], ["id", "nome"])
    df.write.jdbc(
        url="jdbc:IRIS://localhost:1972/DATASPARK",
        dbtable="ft_write_demo",
        user="suser", password="pass123",
        driver="com.intersystems.jdbc.IRISDriver",
        mode="overwrite",
    )
    print("write-back ok")
except Exception as e:
    print("jdbc write not available:", str(e)[:120])

## 8. Reference

See `docs/known_differences.md` for the complete list of behavioral deviations and unsupported APIs.

In [ ]:
if session is not None:
    session.close()
    print("Session closed.")